# Notebook 14 — HeavyHex Circuit Data Generation & MWPM Baseline

**What this notebook does:**  
Loads pre-generated Stim circuits, samples detection events,  
and establishes the MWPM reference decoder for all subsequent notebooks.

**The circuits:**
```
s_stim_circ_z.pickle  ←  Standard Z basis  (use this one first)
s_stim_circ_x.pickle  ←  Standard X basis
stim_circ_z.pickle    ←  Dynamic Z basis   (more complex, deferred)
stim_circ_x.pickle    ←  Dynamic X basis   (more complex, deferred)
```

**Why this circuit is different from all previous notebooks:**
- **Code:** d=5 HeavyHex surface code on 65 physical qubits (IBM Heron r2/r3 topology)
- **Noise:** Real ACES calibration data from ibm_boston — per-qubit, per-gate error rates
  measured from the actual device. Not a simple depolarising model.
- **Detectors:** Non-uniform structure — 12 initial + (9 × 24) + 12 final = 240 total

**Detector structure (Confirmed by inspection):**
```
Block 0 (initial):   12 detectors  — prep round parity checks
Blocks 1–9 (×9):    24 detectors  — per-round syndrome
  ├── First  12: Z-type stabilisers (XOR with previous round)
  └── Second 12: X-type stabilisers (simultaneous weight checks)  
Block 10 (final):   12 detectors  — destructive data qubit readout
```

**MWPM note:** Use `approximate_disjoint_errors=True` when building the DEM,  
as specified — the real noise model produces correlated errors that need this flag.

```bash
pip install stim pymatching numpy
```

In [1]:
import numpy as np
import stim
import pymatching
import pickle
import os

print(f"Stim       : {stim.__version__}")
print(f"PyMatching : {pymatching.__version__}")
print(f"NumPy      : {np.__version__}")

Stim       : 1.15.0
PyMatching : 2.3.1
NumPy      : 2.2.6


---
## 1. Load Circuit

Place the four `.pickle` files in the same directory as this notebook.

In [2]:
CIRCUIT_FILE = "s_stim_circ_z.pickle"   # ← Standard Z basis — focus circuit

def load_circuit_from_pickle(pkl_path: str):
    """
    Load a stim.Circuit from pickle files without calling pickle.load().

    Pickles were saved with a different version of stim than what is
    installed here, which causes an ImportError when using pickle.load() directly.
    This function bypasses the issue by extracting the circuit text from the raw
    bytes and reconstructing the circuit from scratch.

    Works for all four of circuit files:
      s_stim_circ_z.pickle, s_stim_circ_x.pickle,
      stim_circ_z.pickle,   stim_circ_x.pickle
    """
    with open(pkl_path, "rb") as f:
        raw = f.read()

    # The circuit text starts at the first QUBIT_COORDS line
    start = raw.find(b"QUBIT_COORDS")
    if start == -1:
        raise ValueError(f"No QUBIT_COORDS found in {pkl_path}")

    # Walk forward from OBSERVABLE_INCLUDE (always last instruction)
    # stopping at the first non-ASCII byte (pickle footer)
    obs_pos = raw.rfind(b"OBSERVABLE_INCLUDE")
    if obs_pos == -1:
        raise ValueError(f"No OBSERVABLE_INCLUDE found in {pkl_path}")

    end = obs_pos
    while end < len(raw):
        b = raw[end]
        if b == 0x0A:          # newline → include and stop
            end += 1; break
        elif b > 0x7E or (b < 0x20 and b not in (0x09, 0x0D)):
            break              # non-printable ASCII → pickle footer, stop here
        end += 1

    circuit_text = raw[start:end].decode("ascii")
    return stim.Circuit(circuit_text)


circuit = load_circuit_from_pickle(CIRCUIT_FILE)

print(f"Loaded: {CIRCUIT_FILE}")
print(f"  Physical qubits  : {circuit.num_qubits}")
print(f"  Total measurements: {circuit.num_measurements}")
print(f"  Total detectors  : {circuit.num_detectors}")
print(f"  Observables      : {circuit.num_observables}")
print()

assert circuit.num_detectors == 240, f"Expected 240 detectors, got {circuit.num_detectors}"
print("Detector structure (Standard Z):")
print(f"  Initial block   :  12 detectors")
print(f"  Per-round (×9)  :  24 detectors each  (12 Z-type + 12 X-type)")
print(f"  Final block     :  12 detectors")
print(f"  Total           : 240  ✓")
print()
print("This is a d=5 HeavyHex surface code on IBM Heron r2/r3 topology.")
print("Noise model: Real ACES calibration from ibm_boston.")


Loaded: s_stim_circ_z.pickle
  Physical qubits  : 135
  Total measurements: 425
  Total detectors  : 240
  Observables      : 1

Detector structure (Standard Z):
  Initial block   :  12 detectors
  Per-round (×9)  :  24 detectors each  (12 Z-type + 12 X-type)
  Final block     :  12 detectors
  Total           : 240  ✓

This is a d=5 HeavyHex surface code on IBM Heron r2/r3 topology.
Noise model: Real ACES calibration from ibm_boston.


---
## 2. Sample Detection Events

We sample two ways:

In [3]:
NUM_SHOTS = 100_000

# ── Method 1: via raw measurements + m2d converter ────────────────────────────
# This gives us both detection_events AND raw_measurements for use with the LSTM
sampler  = circuit.compile_sampler()
raw_all  = np.array(sampler.sample(shots=NUM_SHOTS), dtype=np.bool_)

converter = circuit.compile_m2d_converter()
detection_events, observable_flips = converter.convert(
    measurements=raw_all,
    separate_observables=True,
)
detection_events = np.array(detection_events, dtype=np.bool_)
observable_flips = np.array(observable_flips, dtype=np.bool_).squeeze()

# Raw measurements — all 425 measurement bits per shot
raw_measurements = raw_all   # (N, 425)

print("Shapes:")
print(f"  raw_all          : {raw_all.shape}         ← all {circuit.num_measurements} measurement bits")
print(f"  detection_events : {detection_events.shape}  ← 240 detectors")
print(f"  observable_flips : {observable_flips.shape}   ← 1 logical observable")
print()
print(f"Logical error rate (trivial): {observable_flips.mean():.4f}  "
      f"({100*observable_flips.mean():.2f}%)")
print()
print("Note: real noise model → much richer error distribution than depolarising.")

Shapes:
  raw_all          : (100000, 425)         ← all 425 measurement bits
  detection_events : (100000, 240)  ← 240 detectors
  observable_flips : (100000,)   ← 1 logical observable

Logical error rate (trivial): 0.4981  (49.81%)

Note: real noise model → much richer error distribution than depolarising.


---
## 3. Build the Correct Temporal Input for LSTM/Transformer

**This is the key correction.** The 240 detectors are NOT a simple `(9, anything)` reshape.  
The correct structure is 11 time steps of 24 features each:

```
Step  0:  detectors[  0: 12] + 12 zeros   (initial, 12 real + 12 padding)
Steps 1-9: detectors[ 12:228] reshaped (9 × 24)  (per-round syndrome)
Step 10:  detectors[228:240] + 12 zeros   (final,   12 real + 12 padding)
```

In [4]:
def build_temporal_input(det_events):
    """
    Convert flat detection events (N, 240) into temporal sequences (N, 11, 24)
    for LSTM/Transformer, correctly respecting the non-uniform detector structure.

    Structure of s_stim_circ_z detectors:
      det[  0: 12] = initial  block (12 detectors, prep round)
      det[ 12:228] = 9 rounds × 24 detectors = 216 detectors
      det[228:240] = final block (12 detectors, readout)

    Returns: (N, 11, 24) float32
      - Step 0:   initial 12 padded to 24 (last 12 = zeros)
      - Steps 1-9: 24 per-round detectors
      - Step 10:  final 12 padded to 24 (last 12 = zeros)
    """
    N = det_events.shape[0]
    assert det_events.shape[1] == 240, f"Expected 240 detectors, got {det_events.shape[1]}"

    seq = np.zeros((N, 11, 24), dtype=np.float32)

    # Step 0: initial 12 detectors, padded to 24
    seq[:, 0, :12] = det_events[:, 0:12].astype(np.float32)

    # Steps 1–9: 9 × 24 per-round detectors
    per_round = det_events[:, 12:228].astype(np.float32)   # (N, 216)
    seq[:, 1:10, :] = per_round.reshape(N, 9, 24)

    # Step 10: final 12 detectors, padded to 24
    seq[:, 10, :12] = det_events[:, 228:240].astype(np.float32)

    return seq


det_temporal = build_temporal_input(detection_events)

print("Temporal input built:")
print(f"  Flat input  : {detection_events.shape}")
print(f"  Temporal    : {det_temporal.shape}  (N, 11 steps, 24 features)")
print()
print("Step content:")
print(f"  Step  0: initial  prep    — {(det_temporal[:,  0, :12] != 0).any(axis=0).sum()} active channels (of 12 real + 12 zeros)")
for r in range(1, 10):
    active = (det_temporal[:, r, :] != 0).any(axis=0).sum()
    print(f"  Step {r:2d}: round {r:2d} syndrome — {active} active channels (all 24 real)")
print(f"  Step 10: final readout    — {(det_temporal[:, 10, :12] != 0).any(axis=0).sum()} active channels (of 12 real + 12 zeros)")

Temporal input built:
  Flat input  : (100000, 240)
  Temporal    : (100000, 11, 24)  (N, 11 steps, 24 features)

Step content:
  Step  0: initial  prep    — 12 active channels (of 12 real + 12 zeros)
  Step  1: round  1 syndrome — 24 active channels (all 24 real)
  Step  2: round  2 syndrome — 24 active channels (all 24 real)
  Step  3: round  3 syndrome — 24 active channels (all 24 real)
  Step  4: round  4 syndrome — 24 active channels (all 24 real)
  Step  5: round  5 syndrome — 24 active channels (all 24 real)
  Step  6: round  6 syndrome — 24 active channels (all 24 real)
  Step  7: round  7 syndrome — 24 active channels (all 24 real)
  Step  8: round  8 syndrome — 24 active channels (all 24 real)
  Step  9: round  9 syndrome — 24 active channels (all 24 real)
  Step 10: final readout    — 12 active channels (of 12 real + 12 zeros)


---
## 4. MWPM Baseline

Using `approximate_disjoint_errors=True` as specified —  
the ACES noise model produces non-decomposable error mechanisms that require this flag.

In [5]:
print("Building DEM and PyMatching object...")
# approximate_disjoint_errors=True is required for the ACES noise model
dem = circuit.detector_error_model(
    decompose_errors=True,
    approximate_disjoint_errors=True
)
matching = pymatching.Matching.from_detector_error_model(dem)
print(f"DEM built.")
print()

# Decode the full dataset
print(f"Decoding {NUM_SHOTS:,} shots with MWPM...")
predicted = matching.decode_batch(
    np.array(detection_events, dtype=np.bool_))

mwpm_errors = np.any(predicted != observable_flips[:, None], axis=1)
mwpm_ler    = float(mwpm_errors.mean())
trivial_ler = float(observable_flips.mean())

print(f"  Trivial LER        : {trivial_ler:.5f}  ({100*trivial_ler:.3f}%)")
print(f"  MWPM LER           : {mwpm_ler:.5f}  ({100*mwpm_ler:.3f}%)")
print(f"  Suppression factor : {trivial_ler/mwpm_ler:.1f}x")
print()
print("Note: MWPM suppression on real ACES noise will be lower than on simple")
print("depolarising noise — the noise is harder and more correlated.")

Building DEM and PyMatching object...
DEM built.

Decoding 100,000 shots with MWPM...
  Trivial LER        : 0.49814  (49.814%)
  MWPM LER           : 0.22693  (22.693%)
  Suppression factor : 2.2x

Note: MWPM suppression on real ACES noise will be lower than on simple
depolarising noise — the noise is harder and more correlated.


---
## 5. Save All Data

In [6]:
os.makedirs("data/heavyhex", exist_ok=True)

# Save detection events (flat + temporal) and raw measurements
np.save("data/heavyhex/detection_events.npy",   detection_events.astype(np.float32))
np.save("data/heavyhex/det_temporal.npy",        det_temporal)
np.save("data/heavyhex/raw_measurements.npy",    raw_measurements.astype(np.float32))
np.save("data/heavyhex/observable_flips.npy",    observable_flips.astype(np.float32))

# Save MWPM result for comparison
os.makedirs("results/heavyhex", exist_ok=True)
np.save("results/heavyhex/mwpm_result.npy",
        np.array([mwpm_ler, trivial_ler]))

print("Saved to data/heavyhex/:")
print(f"  detection_events.npy  : {detection_events.shape}  (flat, for MLP)")
print(f"  det_temporal.npy      : {det_temporal.shape}  (11 steps × 24, for LSTM/Transformer)")
print(f"  raw_measurements.npy  : {raw_measurements.shape}  (all meas bits, for raw-LSTM)")
print(f"  observable_flips.npy  : {observable_flips.shape}")
print()
print("Saved to results/heavyhex/:")
print(f"  mwpm_result.npy  →  MWPM LER={mwpm_ler:.5f}  trivial={trivial_ler:.5f}")
print()
print("Run notebook 15 next — LSTM on the correct 11×24 temporal structure.")

Saved to data/heavyhex/:
  detection_events.npy  : (100000, 240)  (flat, for MLP)
  det_temporal.npy      : (100000, 11, 24)  (11 steps × 24, for LSTM/Transformer)
  raw_measurements.npy  : (100000, 425)  (all meas bits, for raw-LSTM)
  observable_flips.npy  : (100000,)

Saved to results/heavyhex/:
  mwpm_result.npy  →  MWPM LER=0.22693  trivial=0.49814

Run notebook 15 next — LSTM on the correct 11×24 temporal structure.
